In [6]:
import os
import cv2
import csv
import numpy as np
import torch
from PIL import Image
from collections import deque

from torchvision.models.detection import keypointrcnn_resnet50_fpn
from torchvision.models.detection import KeypointRCNN_ResNet50_FPN_Weights

# ---------- paths ----------
video_path = r"D:\MQ\BodyMovement.mp4"
out_video_path = r"D:\MQ\BodyMovement_video_overlay.mp4"
out_npy_path = r"D:\MQ\BodyMovement_video_keypoints.npy"
out_csv_path = r"D:\MQ\BodyMovement_video_keypoints.csv"
out_events_csv_path = r"D:\MQ\BodyMovement_video_events.csv"

# ---------- device ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

# ---------- model ----------
weights = KeypointRCNN_ResNet50_FPN_Weights.DEFAULT
model = keypointrcnn_resnet50_fpn(weights=weights)
model.eval()
model.to(device)

preprocess = weights.transforms()

# ---------- metadata ----------
names = list(weights.meta["keypoint_names"])
name_to_idx = {name: i for i, name in enumerate(names)}
print(name_to_idx)

# ---------- options ----------
score_threshold = 0.8
keep_only_top_person = True
max_frames = None   # use 300 first if you want a quick test

# ---------- thresholds ----------
hand_margin_px = 15                    # wrist above/below shoulder margin
wave_window = 12                       # history length for wave check
wave_min_flips = 3
wave_min_amp_norm = 0.18               # normalized by shoulder width
wave_deadband = 0.015                  # ignore tiny x changes
head_baseline_frames = 10
head_x_thresh = 0.10                   # normalized by shoulder width
head_y_thresh = 0.10                   # normalized by shoulder width
event_banner_frames = 12               # how long new events stay on screen
font = cv2.FONT_HERSHEY_SIMPLEX

using device: cuda
gpu: NVIDIA GeForce GTX 1050 Ti
{'nose': 0, 'left_eye': 1, 'right_eye': 2, 'left_ear': 3, 'right_ear': 4, 'left_shoulder': 5, 'right_shoulder': 6, 'left_elbow': 7, 'right_elbow': 8, 'left_wrist': 9, 'right_wrist': 10, 'left_hip': 11, 'right_hip': 12, 'left_knee': 13, 'right_knee': 14, 'left_ankle': 15, 'right_ankle': 16}


In [7]:
def kp_valid(point):
    return (
        point is not None and
        len(point) >= 3 and
        not np.isnan(point[0]) and
        not np.isnan(point[1]) and
        point[2] > 0
    )

def get_point(kp, name):
    return kp[name_to_idx[name]]

def midpoint(p1, p2):
    return np.array([(p1[0] + p2[0]) / 2.0, (p1[1] + p2[1]) / 2.0], dtype=np.float32)

def shoulder_width(kp):
    ls = get_point(kp, "left_shoulder")
    rs = get_point(kp, "right_shoulder")
    if not (kp_valid(ls) and kp_valid(rs)):
        return np.nan
    return float(np.linalg.norm(ls[:2] - rs[:2]))

def hand_position(kp, side, margin_px=15):
    shoulder = get_point(kp, f"{side}_shoulder")
    wrist = get_point(kp, f"{side}_wrist")

    if not (kp_valid(shoulder) and kp_valid(wrist)):
        return "UNKNOWN"

    if wrist[1] < shoulder[1] - margin_px:
        return "UP"
    if wrist[1] > shoulder[1] + margin_px:
        return "DOWN"
    return "MIDDLE"

def normalized_offsets(kp):
    nose = get_point(kp, "nose")
    ls = get_point(kp, "left_shoulder")
    rs = get_point(kp, "right_shoulder")

    if not (kp_valid(nose) and kp_valid(ls) and kp_valid(rs)):
        return None, None, None

    sw = shoulder_width(kp)
    if np.isnan(sw) or sw < 1e-6:
        return None, None, None

    sm = midpoint(ls, rs)
    dx = float((nose[0] - sm[0]) / sw)
    dy = float((nose[1] - sm[1]) / sw)
    return dx, dy, sw

def head_direction(dx, dy, baseline_dx, baseline_dy, x_thresh=0.10, y_thresh=0.10):
    if dx is None or dy is None or baseline_dx is None or baseline_dy is None:
        return "UNKNOWN", "UNKNOWN"

    # horizontal
    if dx < baseline_dx - x_thresh:
        horiz = "LEFT"
    elif dx > baseline_dx + x_thresh:
        horiz = "RIGHT"
    else:
        horiz = "CENTER"

    # vertical
    if dy < baseline_dy - y_thresh:
        vert = "UP"
    elif dy > baseline_dy + y_thresh:
        vert = "DOWN"
    else:
        vert = "CENTER"

    return horiz, vert

def count_sign_flips(values, deadband=0.0):
    arr = np.asarray(values, dtype=float)
    if len(arr) < 3:
        return 0

    dx = np.diff(arr)
    signs = []
    for d in dx:
        if abs(d) <= deadband:
            continue
        signs.append(np.sign(d))

    if len(signs) < 2:
        return 0

    flips = 0
    for i in range(1, len(signs)):
        if signs[i] * signs[i - 1] < 0:
            flips += 1
    return flips

def wave_flag_from_history(hist, min_flips=3, min_amp_norm=0.18, deadband=0.015):
    if len(hist) < 6:
        return False

    arr = np.asarray(hist, dtype=float)
    amp = float(np.nanmax(arr) - np.nanmin(arr))
    flips = count_sign_flips(arr, deadband=deadband)
    return amp >= min_amp_norm and flips >= min_flips

def draw_text_block(img, lines, x=10, y=25, dy=26, scale=0.65, thickness=2):
    yy = y
    for line in lines:
        cv2.putText(img, line, (x, yy), font, scale, (255, 255, 255), thickness, cv2.LINE_AA)
        yy += dy

In [8]:
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("video opened")
print("fps:", fps)
print("size:", width, height)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(out_video_path, fourcc, fps if fps > 0 else 25.0, (width, height))

all_keypoints = []
all_scores = []

header = ["frame", "time_sec", "det_score"]
for j in range(17):
    header += [f"kp{j}_x", f"kp{j}_y", f"kp{j}_v"]
header += [
    "left_hand_status", "right_hand_status",
    "left_wave_status", "right_wave_status",
    "head_horizontal", "head_vertical"
]

events_header = ["event", "frame", "time_sec", "value"]

counts = {
    "left_hand_raise": 0,
    "right_hand_raise": 0,
    "left_hand_down": 0,
    "right_hand_down": 0,
    "left_wave": 0,
    "right_wave": 0,
    "head_left": 0,
    "head_right": 0,
    "head_up": 0,
    "head_down": 0,
}

# persistent current statuses
current_status = {
    "left_hand": "UNKNOWN",
    "right_hand": "UNKNOWN",
    "left_wave": "IDLE",
    "right_wave": "IDLE",
    "head_horizontal": "UNKNOWN",
    "head_vertical": "UNKNOWN",
}

prev_status = current_status.copy()

left_wrist_hist = deque(maxlen=wave_window)
right_wrist_hist = deque(maxlen=wave_window)

baseline_dx_samples = []
baseline_dy_samples = []
baseline_dx = None
baseline_dy = None

with open(out_csv_path, "w", newline="", encoding="utf-8") as f_kp, \
     open(out_events_csv_path, "w", newline="", encoding="utf-8") as f_ev:

    csv_writer = csv.writer(f_kp)
    csv_writer.writerow(header)

    event_writer = csv.writer(f_ev)
    event_writer.writerow(events_header)

    frame_idx = 0

    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            print("end of video")
            break

        if max_frames is not None and frame_idx >= max_frames:
            print("reached max_frames")
            break

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(frame_rgb)
        x = preprocess(pil_img).to(device)

        with torch.no_grad():
            pred = model([x])[0]

        boxes = pred["boxes"].detach().cpu()
        scores = pred["scores"].detach().cpu()
        keypoints = pred["keypoints"].detach().cpu()

        chosen_kp = np.full((17, 3), np.nan, dtype=np.float32)
        chosen_score = np.nan

        keep = scores >= score_threshold
        kept_idx = torch.where(keep)[0]

        if len(kept_idx) > 0:
            if keep_only_top_person:
                best_local = kept_idx[torch.argmax(scores[kept_idx])].item()
                selected_indices = [best_local]
            else:
                selected_indices = kept_idx.tolist()

            best_idx = selected_indices[0]
            chosen_kp = keypoints[best_idx].numpy().astype(np.float32)
            chosen_score = float(scores[best_idx].item())

            for idx in selected_indices:
                box = boxes[idx].numpy().astype(int)
                kp = keypoints[idx].numpy()

                x1, y1, x2, y2 = box.tolist()
                cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(
                    frame_bgr,
                    f"person {scores[idx].item():.2f}",
                    (x1, max(20, y1 - 8)),
                    font, 0.6, (0, 255, 0), 2, cv2.LINE_AA
                )

                for point in kp:
                    xk, yk, vk = point.tolist()
                    if vk > 0:
                        cv2.circle(frame_bgr, (int(xk), int(yk)), 3, (0, 0, 255), -1)

        all_keypoints.append(chosen_kp)
        all_scores.append(chosen_score)

        time_sec = frame_idx / fps if fps and fps > 0 else 0.0

        # ---------- hand status ----------
        left_hand = hand_position(chosen_kp, "left", margin_px=hand_margin_px)
        right_hand = hand_position(chosen_kp, "right", margin_px=hand_margin_px)

        # ---------- head status ----------
        dx, dy, sw = normalized_offsets(chosen_kp)

        if dx is not None and dy is not None:
            if len(baseline_dx_samples) < head_baseline_frames:
                baseline_dx_samples.append(dx)
                baseline_dy_samples.append(dy)

            if len(baseline_dx_samples) > 0:
                baseline_dx = float(np.median(baseline_dx_samples))
                baseline_dy = float(np.median(baseline_dy_samples))

        head_horizontal, head_vertical = head_direction(
            dx, dy, baseline_dx, baseline_dy,
            x_thresh=head_x_thresh, y_thresh=head_y_thresh
        )

        # ---------- wave status ----------
        ls = get_point(chosen_kp, "left_shoulder")
        rs = get_point(chosen_kp, "right_shoulder")
        sm = midpoint(ls, rs) if (kp_valid(ls) and kp_valid(rs)) else None

        lw = get_point(chosen_kp, "left_wrist")
        rw = get_point(chosen_kp, "right_wrist")

        sw_now = shoulder_width(chosen_kp)

        if sm is not None and not np.isnan(sw_now) and sw_now > 1e-6:
            if left_hand == "UP" and kp_valid(lw):
                left_wrist_hist.append(float((lw[0] - sm[0]) / sw_now))
            else:
                left_wrist_hist.clear()

            if right_hand == "UP" and kp_valid(rw):
                right_wrist_hist.append(float((rw[0] - sm[0]) / sw_now))
            else:
                right_wrist_hist.clear()
        else:
            left_wrist_hist.clear()
            right_wrist_hist.clear()

        left_wave = "ACTIVE" if wave_flag_from_history(
            left_wrist_hist,
            min_flips=wave_min_flips,
            min_amp_norm=wave_min_amp_norm,
            deadband=wave_deadband
        ) else "IDLE"

        right_wave = "ACTIVE" if wave_flag_from_history(
            right_wrist_hist,
            min_flips=wave_min_flips,
            min_amp_norm=wave_min_amp_norm,
            deadband=wave_deadband
        ) else "IDLE"

        # ---------- only update stored statuses when changed ----------
        current_status["left_hand"] = left_hand
        current_status["right_hand"] = right_hand
        current_status["left_wave"] = left_wave
        current_status["right_wave"] = right_wave
        current_status["head_horizontal"] = head_horizontal
        current_status["head_vertical"] = head_vertical

        # ---------- count only on transitions ----------
        # left hand
        if prev_status["left_hand"] != current_status["left_hand"]:
            if current_status["left_hand"] == "UP":
                counts["left_hand_raise"] += 1
                event_writer.writerow(["left_hand_raise", frame_idx, time_sec, "UP"])
            elif current_status["left_hand"] == "DOWN":
                counts["left_hand_down"] += 1
                event_writer.writerow(["left_hand_down", frame_idx, time_sec, "DOWN"])

        # right hand
        if prev_status["right_hand"] != current_status["right_hand"]:
            if current_status["right_hand"] == "UP":
                counts["right_hand_raise"] += 1
                event_writer.writerow(["right_hand_raise", frame_idx, time_sec, "UP"])
            elif current_status["right_hand"] == "DOWN":
                counts["right_hand_down"] += 1
                event_writer.writerow(["right_hand_down", frame_idx, time_sec, "DOWN"])

        # left wave
        if prev_status["left_wave"] != current_status["left_wave"] and current_status["left_wave"] == "ACTIVE":
            counts["left_wave"] += 1
            event_writer.writerow(["left_wave", frame_idx, time_sec, "ACTIVE"])

        # right wave
        if prev_status["right_wave"] != current_status["right_wave"] and current_status["right_wave"] == "ACTIVE":
            counts["right_wave"] += 1
            event_writer.writerow(["right_wave", frame_idx, time_sec, "ACTIVE"])

        # head horizontal
        if prev_status["head_horizontal"] != current_status["head_horizontal"]:
            if current_status["head_horizontal"] == "LEFT":
                counts["head_left"] += 1
                event_writer.writerow(["head_left", frame_idx, time_sec, "LEFT"])
            elif current_status["head_horizontal"] == "RIGHT":
                counts["head_right"] += 1
                event_writer.writerow(["head_right", frame_idx, time_sec, "RIGHT"])

        # head vertical
        if prev_status["head_vertical"] != current_status["head_vertical"]:
            if current_status["head_vertical"] == "UP":
                counts["head_up"] += 1
                event_writer.writerow(["head_up", frame_idx, time_sec, "UP"])
            elif current_status["head_vertical"] == "DOWN":
                counts["head_down"] += 1
                event_writer.writerow(["head_down", frame_idx, time_sec, "DOWN"])

        prev_status = current_status.copy()

        # ---------- clean overlay ----------
        lines = [
            f"frame: {frame_idx}",
            f"time: {time_sec:.2f}s",
            f"score: {chosen_score:.3f}" if not np.isnan(chosen_score) else "score: nan",
            "",
            f"left hand:  {current_status['left_hand']}",
            f"right hand: {current_status['right_hand']}",
            f"left wave:  {current_status['left_wave']}",
            f"right wave: {current_status['right_wave']}",
            f"head horiz: {current_status['head_horizontal']}",
            f"head vert:  {current_status['head_vertical']}",
            "",
            f"left raise count:  {counts['left_hand_raise']}",
            f"right raise count: {counts['right_hand_raise']}",
            f"left down count:   {counts['left_hand_down']}",
            f"right down count:  {counts['right_hand_down']}",
            f"left wave count:   {counts['left_wave']}",
            f"right wave count:  {counts['right_wave']}",
            f"head left count:   {counts['head_left']}",
            f"head right count:  {counts['head_right']}",
            f"head up count:     {counts['head_up']}",
            f"head down count:   {counts['head_down']}",
        ]

        draw_text_block(frame_bgr, lines, x=10, y=25, dy=24, scale=0.60, thickness=2)

        row = [frame_idx, time_sec, chosen_score] + chosen_kp.reshape(-1).tolist() + [
            current_status["left_hand"],
            current_status["right_hand"],
            current_status["left_wave"],
            current_status["right_wave"],
            current_status["head_horizontal"],
            current_status["head_vertical"],
        ]
        csv_writer.writerow(row)
        writer.write(frame_bgr)

        if frame_idx % 10 == 0:
            print("processed frame", frame_idx)

        frame_idx += 1

cap.release()
writer.release()

all_keypoints = np.stack(all_keypoints, axis=0)
np.save(out_npy_path, all_keypoints)

print("done")
print("saved video:", out_video_path)
print("saved npy:", out_npy_path, all_keypoints.shape)
print("saved csv:", out_csv_path)
print("saved events csv:", out_events_csv_path)
print("final counts:", counts)

video opened
fps: 59.96256704410012
size: 478 850
processed frame 0
processed frame 10
processed frame 20
processed frame 30
processed frame 40
processed frame 50
processed frame 60
processed frame 70
processed frame 80
processed frame 90
processed frame 100
processed frame 110
processed frame 120
processed frame 130
processed frame 140
processed frame 150
processed frame 160
processed frame 170
processed frame 180
processed frame 190
processed frame 200
processed frame 210
processed frame 220
processed frame 230
processed frame 240
processed frame 250
processed frame 260
processed frame 270
processed frame 280
processed frame 290
processed frame 300
processed frame 310
processed frame 320
processed frame 330
processed frame 340
processed frame 350
processed frame 360
processed frame 370
processed frame 380
processed frame 390
processed frame 400
processed frame 410
processed frame 420
processed frame 430
processed frame 440
processed frame 450
processed frame 460
processed frame 470
p

In [9]:
import csv
from collections import Counter

with open(out_events_csv_path, "r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(rows[:20])
print()

counts = Counter(row["event"] for row in rows)
print(counts)

print()
print("Open this file to review the overlayed result:")
print(out_video_path)

[{'event': 'left_hand_down', 'frame': '0', 'time_sec': '0.0', 'value': 'DOWN'}, {'event': 'right_hand_down', 'frame': '0', 'time_sec': '0.0', 'value': 'DOWN'}, {'event': 'head_right', 'frame': '51', 'time_sec': '0.850530631260191', 'value': 'RIGHT'}, {'event': 'head_up', 'frame': '55', 'time_sec': '0.9172389160649119', 'value': 'UP'}, {'event': 'head_up', 'frame': '58', 'time_sec': '0.9672701296684525', 'value': 'UP'}, {'event': 'head_up', 'frame': '60', 'time_sec': '1.000624272070813', 'value': 'UP'}, {'event': 'head_up', 'frame': '62', 'time_sec': '1.0339784144731734', 'value': 'UP'}, {'event': 'head_left', 'frame': '132', 'time_sec': '2.2013733985557886', 'value': 'LEFT'}, {'event': 'right_hand_raise', 'frame': '325', 'time_sec': '5.42004814038357', 'value': 'UP'}, {'event': 'head_right', 'frame': '447', 'time_sec': '7.454650826927557', 'value': 'RIGHT'}, {'event': 'head_down', 'frame': '447', 'time_sec': '7.454650826927557', 'value': 'DOWN'}, {'event': 'head_left', 'frame': '448', 